# YOLO pose filtering on the full dataset (no labels)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

## Parameters

In [ ]:
PATH_MODEL = "path/to/best.pt"
PATH_IMAGES = "path/to/full/dataset/images"

# CSV produced right after inference and reloaded before plotting
CSV_PATH = "results_full.csv"

CONF_THRESHOLD = 0.5      # traitable (>=) vs non traitable (<)
IMG_CONF = 0.25           # detector confidence to keep a detection
DEVICE = None             # e.g. 0, "cpu", or None for automatic

## Helper

In [ ]:
def image_confidence(kpts_conf):
    # mean confidence of all detected keypoints (0 if no detection)
    if kpts_conf.size == 0:
        return 0.0
    return float(np.mean(kpts_conf))

## Inference

In [ ]:
def run_inference():
    model = YOLO(PATH_MODEL)
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    image_files = sorted(
        f for f in glob.glob(os.path.join(PATH_IMAGES, "*"))
        if f.lower().endswith(exts)
    )
    predict_kwargs = dict(conf=IMG_CONF, verbose=False)
    if DEVICE is not None:
        predict_kwargs["device"] = DEVICE

    rows = []
    for img_path in image_files:
        res = model.predict(img_path, **predict_kwargs)[0]
        if res.keypoints is not None and res.keypoints.data.shape[0] > 0:
            kconf = res.keypoints.data.cpu().numpy()[:, :, 2]
        else:
            kconf = np.zeros((0, 0))
        rows.append({"image": os.path.basename(img_path),
                     "confidence": image_confidence(kconf)})
    return pd.DataFrame(rows)

In [ ]:
# Run inference once and store per-image confidence into a CSV.
if not os.path.exists(CSV_PATH):
    df = run_inference()
    df.to_csv(CSV_PATH, index=False)
    print("Inference done, saved:", CSV_PATH)
else:
    print("CSV already present, skipping inference:", CSV_PATH)

## Load results

In [ ]:
df = pd.read_csv(CSV_PATH)
print(len(df), "images")
df.head()

## Counts at the chosen threshold

In [ ]:
conf = df["confidence"].values
total = len(conf)
n_proc = int(np.sum(conf >= CONF_THRESHOLD))
n_non = int(np.sum(conf < CONF_THRESHOLD))
print(f"traitable:     {n_proc} ({100 * n_proc / total:.1f}%)")
print(f"non traitable: {n_non} ({100 * n_non / total:.1f}%)")

## Class proportions vs confidence threshold (step 0.01)

In [ ]:
thrs = np.round(np.arange(0.0, 1.0 + 1e-9, 0.01), 2)
pct_proc = np.array([100 * np.mean(conf >= t) for t in thrs])
pct_non = 100 - pct_proc

plt.figure()
plt.plot(thrs, pct_proc, label="traitable")
plt.plot(thrs, pct_non, label="non traitable")
plt.xlabel("confidence threshold")
plt.ylabel("% of images")
plt.title("Image distribution vs confidence threshold")
plt.legend()
plt.show()